# Stage C — Uncapped retraining of PatchTST and DLinear (POST HOC sensitivity analysis)

**Why:** the frozen runs stopped at the 50-epoch limit while validation loss was still improving: DLinear at 1, 6, 24 h and PatchTST at 6 and 24 h (25 jobs). 1-h PatchTST early-stopped normally (best epochs 15–29) and is **not** retrained.

**Protocol — identical to your frozen Step 10D training loop except one setting:**

| Setting | Frozen | Stage C |
|---|---|---|
| Max epochs | 50 | **300** (safety ceiling; early stopping decides) |
| Everything else (data, scaler, builder, Adam lr 1e-4, batch 64, seeded shuffling, MSE on target channels, patience 10, restore best, seeds) | — | unchanged |

**Built-in reproduction checks (no extra training cost):** with the same seed, the first 50 epochs replay the frozen run. The notebook therefore (i) compares the epoch 1–50 validation curve with your frozen history, and (ii) keeps the best-within-50-epochs weights and checks their test predictions against your frozen predictions. Small GPU non-determinism (mainly PatchTST attention kernels) is reported, not hidden.

**Writes only to** `Benchmark_V5/IJGE_Revision/StageC_uncapped/`. **Resumable:** completed jobs are skipped on re-run (Colab disconnects are fine). Expected runtime on A100: roughly 2–5 h in total (most of it PatchTST at 6 h and 24 h).

In [ ]:
# ===================== C0 — SETUP =====================
from google.colab import drive
drive.mount('/content/gdrive')

import os, sys, json, hashlib, gc, time, random, shutil, zipfile, importlib
from pathlib import Path
from types import SimpleNamespace
import numpy as np, pandas as pd

MYDRIVE = Path('/content/gdrive/MyDrive')
BENCH   = MYDRIVE / 'Benchmark_V5'
OUT     = BENCH / 'IJGE_Revision' / 'StageC_uncapped'
OUT.mkdir(parents=True, exist_ok=True)

SEEDS = [1, 7, 21, 42, 100]
N_TEST = 6576
TARGET_MEAN  = np.array([55821.81814021, 4184.23949544], dtype=np.float64)   # frozen V6 scaler
TARGET_SCALE = np.array([10034.61850806, 6415.36095093], dtype=np.float64)
DATA_SHA = {1: 'ed67feaed54517cac5f84d314caca75a7b3837ed9bb7f17c7613f0023bd69912',
            6: '8b87dd4eac8f8690e8f800b35a3549327783f8886deb89e285511f4c32b5653a',
            24: 'c4932ec4bda1b0608ac38a4456ccdc896059c281b90d51d9ee631e92eed55936'}
DATA_PATH = {1: BENCH / 'data' / 'frozen_168h_sequences.npz',
             6: BENCH / 'MultiHorizon_V6' / 'datasets' / 'frozen_168h_h6_v6.npz',
             24: BENCH / 'MultiHorizon_V6' / 'datasets' / 'frozen_168h_h24_v6.npz'}
PATCH_ROOT = BENCH / 'external_models' / 'PatchTST' / 'PatchTST_supervised'

def sha256(p, chunk=1 << 20):
    h = hashlib.sha256()
    with open(p, 'rb') as f:
        for b in iter(lambda: f.read(chunk), b''): h.update(b)
    return h.hexdigest()

def guard_write(p):
    p = Path(p).resolve()
    assert str(p).startswith(str(OUT.resolve())), f'REFUSING to write outside OUT: {p}'
    p.parent.mkdir(parents=True, exist_ok=True)
    return p

def inverse_targets(y):
    return np.asarray(y, np.float64) * TARGET_SCALE + TARGET_MEAN

def rmse(e): return float(np.sqrt(np.mean(np.asarray(e, np.float64) ** 2)))

import torch, torch.nn as nn
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', DEVICE, '| torch', torch.__version__, '| numpy', np.__version__)
print('Write location:', OUT)


In [ ]:
# ===================== Exact certified PyTorch builders (copied from your Step 10D-1A cells) =====================
assert PATCH_ROOT.exists(), PATCH_ROOT
for m in [k for k in sys.modules if k == 'models' or k.startswith('models.')]: del sys.modules[m]
if str(PATCH_ROOT) not in sys.path: sys.path.insert(0, str(PATCH_ROOT))
PatchTSTModel = importlib.import_module('models.PatchTST').Model

def build_patchtst():
    configs = SimpleNamespace(seq_len=168, pred_len=1, enc_in=8, e_layers=2, n_heads=4, d_model=64, d_ff=128,
                              dropout=0.2, patch_len=12, stride=12, revin=1, affine=1, subtract_last=0,
                              decomposition=0, kernel_size=25, individual=0, padding_patch='end',
                              fc_dropout=0.0, head_dropout=0.0)
    model = PatchTSTModel(configs, max_seq_len=1024, d_k=None, d_v=None, norm='BatchNorm', attn_dropout=0.0,
                          act='gelu', key_padding_mask='auto', padding_var=None, attn_mask=None, res_attention=True,
                          pre_norm=False, store_attn=False, pe='zeros', learn_pe=True, pretrain_head=False,
                          head_type='flatten', verbose=False)
    assert sum(p.numel() for p in model.parameters()) == 69715
    assert sum(p.numel() for p in model.parameters() if p.requires_grad) == 69713
    return model

class MovingAvg(nn.Module):
    def __init__(self, kernel_size, stride=1):
        super().__init__(); self.kernel_size = kernel_size
        self.avg = nn.AvgPool1d(kernel_size=kernel_size, stride=stride, padding=0)
    def forward(self, x):
        front = x[:, 0:1, :].repeat(1, (self.kernel_size - 1) // 2, 1)
        end = x[:, -1:, :].repeat(1, (self.kernel_size - 1) // 2, 1)
        x = torch.cat([front, x, end], dim=1)
        return self.avg(x.permute(0, 2, 1)).permute(0, 2, 1)

class SeriesDecomp(nn.Module):
    def __init__(self, kernel_size):
        super().__init__(); self.moving_avg = MovingAvg(kernel_size, stride=1)
    def forward(self, x):
        mm = self.moving_avg(x); return x - mm, mm

class DLinearExact(nn.Module):
    def __init__(self, seq_len=168, pred_len=1, enc_in=8, kernel_size=25, individual=False):
        super().__init__()
        assert not individual
        self.decomposition = SeriesDecomp(kernel_size)
        self.Linear_Seasonal = nn.Linear(seq_len, pred_len)
        self.Linear_Trend = nn.Linear(seq_len, pred_len)
    def forward(self, x):
        s, t = self.decomposition(x)
        out = self.Linear_Seasonal(s.permute(0, 2, 1)) + self.Linear_Trend(t.permute(0, 2, 1))
        return out.permute(0, 2, 1)

def route_torch_output(raw):          # frozen PatchTST/DLinear output [B,1,8] -> target channels 0:2
    assert raw.ndim == 3 and raw.shape[1] == 1 and raw.shape[2] == 8
    return raw[:, 0, 0:2]

def load_state_flexible(model, state_raw):
    try:
        r = model.load_state_dict(state_raw, strict=True)
    except RuntimeError:
        assert all(k.startswith('model.') for k in state_raw)
        r = model.load_state_dict({k[len('model.'):]: v for k, v in state_raw.items()}, strict=True)
    assert not r.missing_keys and not r.unexpected_keys
    return model

def predict_torch(model, X, bs=512):
    model.eval(); outs = []
    with torch.no_grad():
        for i in range(0, len(X), bs):
            outs.append(route_torch_output(model(torch.from_numpy(X[i:i+bs]).float().to(DEVICE))).cpu().numpy())
    return np.concatenate(outs).astype(np.float64)
print('PyTorch builders ready.')


In [ ]:
# ===================== C1 — DATA, FROZEN REFERENCES, JOB LIST =====================
import tensorflow as tf        # only for set_all_seeds parity with the frozen trainer
for _g in tf.config.list_physical_devices('GPU'):
    tf.config.experimental.set_memory_growth(_g, True)   # let TF and PyTorch share the GPU
from torch.utils.data import TensorDataset, DataLoader
MAX_EPOCHS, PATIENCE, LR, BATCH = 300, 10, 1e-4, 64
REPLAY_EPOCHS = 50
SHAPES = {1: 34121, 6: 34096, 24: 34006}

def load_data(h):
    assert sha256(DATA_PATH[h]) == DATA_SHA[h], f'dataset hash mismatch h={h}'
    with np.load(DATA_PATH[h], allow_pickle=False) as z:
        d = {k: np.asarray(z[k], dtype=np.float32) for k in ['X_train', 'y_train', 'X_val', 'y_val', 'X_test', 'y_test']}
    assert d['X_train'].shape == (SHAPES[h], 168, 8) and d['X_val'].shape == (8760, 168, 8) and d['X_test'].shape == (N_TEST, 168, 8)
    assert all(np.isfinite(v).all() for v in d.values())
    return d

def frozen_history(h, model, s):
    if h == 1:
        assert model == 'DLinear'
        return pd.read_csv(BENCH / 'results/DLinear/histories' / f'dlinear_seed_{s}_history.csv')
    return pd.read_csv(BENCH / 'MultiHorizon_V6/definitive_training/histories' / f'h{h:02d}_seed{s:03d}_{model}_history.csv')

def frozen_test_pred(h, model, s):
    if h == 1:
        return np.load(BENCH / 'Reliability_V6/degradation_results/pytorch_models/pytorch_reliability_clean_predictions.npz')[f'{model}_seed_{s}_pred_mw']
    return np.load(BENCH / 'MultiHorizon_V6/definitive_training/predictions' / f'h{h:02d}_seed{s:03d}_{model}_predictions.npz')['y_pred_physical']

JOBS = [(h, 'DLinear', s) for h in [1, 6, 24] for s in SEEDS] + [(h, 'PatchTST', s) for h in [6, 24] for s in SEEDS]
for h, m, s in JOBS:      # fail fast if any reference is missing
    frozen_history(h, m, s); frozen_test_pred(h, m, s)
print('Jobs:', len(JOBS), '| all frozen references found')


In [ ]:
# ===================== C2 — TRAINER (frozen loop, max_epochs raised) =====================
def set_all_seeds(seed):                     # identical to frozen Step 10D
    os.environ['PYTHONHASHSEED'] = str(seed); random.seed(seed); np.random.seed(seed)
    tf.keras.utils.set_random_seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)

def make_loaders(d, seed):
    g = torch.Generator(); g.manual_seed(seed)
    pin = torch.cuda.is_available()
    tr = DataLoader(TensorDataset(torch.from_numpy(d['X_train']), torch.from_numpy(d['y_train'])), batch_size=BATCH,
                    shuffle=True, generator=g, num_workers=0, pin_memory=pin, drop_last=False)
    va = DataLoader(TensorDataset(torch.from_numpy(d['X_val']), torch.from_numpy(d['y_val'])), batch_size=BATCH,
                    shuffle=False, num_workers=0, pin_memory=pin, drop_last=False)
    return tr, va

def run_job(h, model_name, seed, d):
    jid = f'h{h:02d}_seed{seed:03d}_{model_name}'
    done = OUT / 'jobs' / jid / 'COMPLETED.json'
    if done.exists():
        print(f'{jid}: SKIP (complete)'); return
    set_all_seeds(seed)
    tr, va = make_loaders(d, seed)
    model = (build_patchtst() if model_name == 'PatchTST' else DLinearExact()).to(DEVICE)
    crit = torch.nn.MSELoss(); opt = torch.optim.Adam(model.parameters(), lr=LR)
    best, best_ep, bad = float('inf'), None, 0
    best50, best50_ep, state50 = float('inf'), None, None
    hist = []; t0 = time.time(); stop = 'max_epochs'
    for ep in range(1, MAX_EPOCHS + 1):
        model.train(); sse = n = 0
        for xb, yb in tr:
            xb, yb = xb.to(DEVICE, non_blocking=True), yb.to(DEVICE, non_blocking=True)
            opt.zero_grad(set_to_none=True)
            loss = crit(route_torch_output(model(xb)), yb); assert torch.isfinite(loss)
            loss.backward(); opt.step()
            sse += float(loss.detach().cpu()) * yb.numel(); n += yb.numel()
        tr_mse = sse / n
        model.eval(); sse = n = 0
        with torch.no_grad():
            for xb, yb in va:
                xb, yb = xb.to(DEVICE, non_blocking=True), yb.to(DEVICE, non_blocking=True)
                loss = crit(route_torch_output(model(xb)), yb)
                sse += float(loss.detach().cpu()) * yb.numel(); n += yb.numel()
        va_mse = sse / n
        if va_mse < best:
            best, best_ep, bad = va_mse, ep, 0
            state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        else:
            bad += 1
        if ep <= REPLAY_EPOCHS and va_mse < best50:
            best50, best50_ep = va_mse, ep
            state50 = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        hist.append([ep, tr_mse, va_mse, best, bad])
        if bad >= PATIENCE:
            stop = 'early_stopping'; break
    secs = time.time() - t0
    jd = OUT / 'jobs' / jid
    torch.save({'model_state_dict': state, 'best_epoch': best_ep}, guard_write(jd / 'best_uncapped.pt'))
    torch.save({'model_state_dict': state50, 'best_epoch': best50_ep}, guard_write(jd / 'best_within_50.pt'))
    H = pd.DataFrame(hist, columns=['epoch', 'train_mse', 'val_mse', 'best_val_so_far', 'bad_epochs'])
    H.to_csv(guard_write(jd / 'history.csv'), index=False)
    # test predictions for both checkpoints
    out = {}
    for tag, st in [('uncapped', state), ('within50', state50)]:
        mm = (build_patchtst() if model_name == 'PatchTST' else DLinearExact())
        mm.load_state_dict(st, strict=True); mm.to(DEVICE)
        out[tag] = inverse_targets(predict_torch(mm, d['X_test']))
    y = inverse_targets(d['y_test'])
    np.savez_compressed(guard_write(jd / 'test_predictions_MW.npz'), y_true=y, pred_uncapped=out['uncapped'], pred_within50=out['within50'])
    # reproduction diagnostics vs frozen
    fh = frozen_history(h, model_name, seed)
    k = min(REPLAY_EPOCHS, len(fh), len(H))
    curve_rel = float(np.max(np.abs(H.val_mse.values[:k] - fh.val_mse.values[:k]) / fh.val_mse.values[:k]))
    fp = np.asarray(frozen_test_pred(h, model_name, seed), np.float64).reshape(N_TEST, 2)
    rec = dict(job=jid, horizon=h, model=model_name, seed=seed, stop_reason=stop, epochs_run=len(H),
               best_epoch_uncapped=best_ep, best_val_uncapped=best, best_epoch_within50=best50_ep, best_val_within50=best50,
               frozen_best_val=float(fh.val_mse.min()), replay_val_curve_max_rel_diff=curve_rel,
               within50_vs_frozen_pred_max_abs_MW=float(np.max(np.abs(out['within50'] - fp))),
               rmse_frozen_load=rmse(fp[:, 0] - y[:, 0]), rmse_frozen_solar=rmse(fp[:, 1] - y[:, 1]),
               rmse_within50_load=rmse(out['within50'][:, 0] - y[:, 0]), rmse_within50_solar=rmse(out['within50'][:, 1] - y[:, 1]),
               rmse_uncapped_load=rmse(out['uncapped'][:, 0] - y[:, 0]), rmse_uncapped_solar=rmse(out['uncapped'][:, 1] - y[:, 1]),
               train_seconds=secs, max_epochs=MAX_EPOCHS, patience=PATIENCE, lr=LR, batch=BATCH,
               dataset_sha256=DATA_SHA[h], torch=torch.__version__, device=str(DEVICE))
    json.dump(rec, open(guard_write(done), 'w'), indent=2)
    print(f"{jid}: {stop} at epoch {len(H)} (best {best_ep}) | replay curve max rel diff {curve_rel:.2e} | "
          f"RMSE load {rec['rmse_frozen_load']:.1f} -> {rec['rmse_uncapped_load']:.1f}, solar {rec['rmse_frozen_solar']:.1f} -> {rec['rmse_uncapped_solar']:.1f} | {secs/60:.1f} min")
print('Trainer ready.')


In [ ]:
# ===================== C3 — RUN ALL 25 JOBS (resumable; re-run this cell after any disconnect) =====================
for h in [1, 6, 24]:
    hj = [j for j in JOBS if j[0] == h]
    if all((OUT / 'jobs' / f'h{h:02d}_seed{s:03d}_{m}' / 'COMPLETED.json').exists() for _, m, s in hj):
        print(f'h={h}: all complete'); continue
    d = load_data(h)
    for _, m, s in hj:
        run_job(h, m, s, d)
    del d; gc.collect()
print('ALL JOBS COMPLETE')


In [ ]:
# ===================== C4 — REPRODUCTION SUMMARY + BUNDLE =====================
R = pd.DataFrame([json.load(open(p)) for p in sorted((OUT / 'jobs').glob('*/COMPLETED.json'))])
assert len(R) == 25, f'{len(R)} of 25 jobs complete'
pd.set_option('display.width', 250); pd.set_option('display.max_columns', 30)
print('Reproduction of frozen runs (first 50 epochs replayed):')
print(R[['job', 'replay_val_curve_max_rel_diff', 'within50_vs_frozen_pred_max_abs_MW', 'rmse_frozen_load', 'rmse_within50_load',
         'rmse_frozen_solar', 'rmse_within50_solar']].round(4).to_string(index=False))
print('\nTraining outcome:')
print(R[['job', 'stop_reason', 'epochs_run', 'best_epoch_uncapped', 'best_val_within50', 'best_val_uncapped']].to_string(index=False))
agg = R.groupby(['horizon', 'model'])[['rmse_frozen_load', 'rmse_uncapped_load', 'rmse_frozen_solar', 'rmse_uncapped_solar']].agg(['mean', 'std']).round(3)
print('\nFive-seed RMSE (MW), frozen vs uncapped:'); print(agg.to_string())
R.to_csv(guard_write(OUT / 'stageC_job_summary.csv'), index=False)
zp = guard_write(OUT / 'StageC_bundle.zip')
with zipfile.ZipFile(zp, 'w', zipfile.ZIP_DEFLATED) as zf:
    zf.write(OUT / 'stageC_job_summary.csv', 'stageC_job_summary.csv')
    for p in (OUT / 'jobs').glob('*/*'):
        if p.suffix in {'.csv', '.json', '.npz'}: zf.write(p, str(p.relative_to(OUT)))
print(f'\nBundle: {zp} ({zp.stat().st_size/1e6:.1f} MB)')
